In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import xarray as xr
from datetime import datetime

In [4]:
# Information about the data locations
rootDir = "/cyfast/adelhass/models/nemo/out/eORCA1_ELIC_6_ref3/"

In [12]:
# Define regions with latitude and longitude bounds
# "The five sectors of Raphael and Hobbs (2014): East Antarctica (71–163◦ E), Ross/Amundsen (163–250◦ E), Amundsen/Bellingshausen (250–293◦ E), Weddell (293–346◦ E), and King Hakon VII (346–71◦ E)."
regions = {
    "Southern Ocean": {"lat_min": -90, "lat_max": 0, "lon_min": -180, "lon_max": 180},
    "East Antarctica": {"lat_min": -90, "lat_max": 0, "lon_min": 71, "lon_max": 163},
    "Ross/Amundsen": {"lat_min": -90, "lat_max": 0, "lon_min": 163, "lon_max": -110},
    "Amundsen/Bellingshausen": {"lat_min": -90, "lat_max": 0, "lon_min": -110, "lon_max": -67},
    "Weddell Sea": {"lat_min": -90, "lat_max": 0, "lon_min": -67, "lon_max": -14},
    "King Hakon VII Sea": {"lat_min": -90, "lat_max": 0, "lon_min": -14, "lon_max": 71},
}

In [2]:
# Information about experiments. This is a dictionary where keys are experiment names
# and values are lists containing a description, start date, and end date.
expinfo = {
    "as01": ["SIC data assimilation experiment", datetime(1970, 1, 1), datetime(2023, 12, 31)],
}


In [13]:
# Loading the data.
exp="as01"
yearb=expinfo[exp][1].year
yeare=expinfo[exp][2].year  
fileIn = rootDir + exp + "/" + exp + "_ensmean_siconc,sithic,sivolu_" + str(yearb) + "-" + str(yeare) + ".nc"

# Test file existence
import os
if not os.path.isfile(fileIn):
    raise FileNotFoundError(f"File not found: {fileIn}")
else:
    print(f"File found: {fileIn}")

ds = xr.open_dataset(fileIn)
print(ds)

File found: /cyfast/adelhass/models/nemo/out/eORCA1_ELIC_6_ref3/as01/as01_ensmean_siconc,sithic,sivolu_1970-2023.nc
<xarray.Dataset> Size: 932MB
Dimensions:            (time_counter: 648, bnds: 2, y: 331, x: 360, nvertex: 4)
Coordinates:
  * time_counter       (time_counter) datetime64[ns] 5kB 1970-01-16T12:00:00 ...
    nav_lon            (y, x) float32 477kB ...
    nav_lat            (y, x) float32 477kB ...
Dimensions without coordinates: bnds, y, x, nvertex
Data variables:
    time_counter_bnds  (time_counter, bnds) datetime64[ns] 10kB ...
    nav_lon_bnds       (y, x, nvertex) float32 2MB ...
    nav_lat_bnds       (y, x, nvertex) float32 2MB ...
    cell_area          (y, x) float32 477kB ...
    siconc             (time_counter, y, x) float32 309MB ...
    sithic             (time_counter, y, x) float32 309MB ...
    sivolu             (time_counter, y, x) float32 309MB ...
Attributes: (12/15)
    CDI:          Climate Data Interface version 2.1.1 (https://mpimet.mpg.de...
    

In [ ]:
# Load the sea ice volu- sivolu  variable
sivolu = ds["sivolu"]

# Load the grid cell area variable
areacello = ds["cell_area"]

# Load the latitude and longitude variables
lat = ds["nav_lat"]
lon = ds["nav_lon"]
# Center longitudes if necessary (-180 to 180)
lon = ((lon + 180) % 360) - 180

# Multiply sea ice volume by grid cell area to get total sea ice volume per grid cell
sivolu_total = sivolu * areacello


In [15]:
# Loop over regions and calculate total sea ice volume
region_volumes = {}
for region_name, bounds in regions.items():
    print(f"Processing region: {region_name}")
    lat_min = bounds["lat_min"]
    lat_max = bounds["lat_max"]
    lon_min = bounds["lon_min"]
    lon_max = bounds["lon_max"]

    # Create a mask for the region
    lat_mask = (lat >= lat_min) & (lat <= lat_max)
    if lon_min < lon_max: # This is to deal with regions crossing the dateline
        lon_mask = (lon >= lon_min) & (lon <= lon_max)
    else:
        lon_mask = (lon >= lon_min) | (lon <= lon_max)

    region_mask = lat_mask & lon_mask

    # Apply the mask to the sea ice volume data
    sivolu_region = sivolu_total.where(region_mask, drop=True)

    # Sum over latitude and longitude to get total volume for the region
    total_volume = sivolu_region.sum(dim=["y", "x"])
    
    # Store the result
    region_volumes[region_name] = total_volume

Processing region: Southern Ocean
Processing region: East Antarctica
Processing region: Ross/Amundsen
Processing region: Amundsen/Bellingshausen
Processing region: Weddell Sea
Processing region: King Hakon VII Sea


In [16]:
# Plot the raw time series for each region
for region_name, total_volume in region_volumes.items():
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    total_volume.plot(ax=ax)
    ax.set_title(f"Total Sea Ice Volume in {region_name} ({exp})")
    ax.set_ylabel("Sea Ice Volume (m³)")
    ax.set_xlabel("Time")
    # Create a safe filename by replacing spaces, slashes with underscores
    safe_region_name = region_name.replace(" ", "_").replace("/", "_")
    fig.savefig(f"sivolu_{safe_region_name}_{exp}.png")